# YOLO11n 3개 물병 데이터셋 실험

**목표**: 동일한 YOLO11n 출발 모델을 3개의 데이터셋에서 각각 학습하고, 각 모델을 세 Test 데이터셋에서 평가(총 9회)합니다.

**생성 결과**:
- Self-Test Performance 그래프
- Cross-Dataset mAP50-95 Heatmap
- 9개 평가 결과 CSV
- `best_mine.pt`, `best_sanghyeon.pt`, `best_jian.pt`

**실행 전**: Colab 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU**를 선택하세요. 세 데이터셋의 ZIP 파일을 준비하세요. 기존 `best.pt`를 따로 업로드할 필요는 없습니다.

*주의: 세 결과를 공정하게 비교하기 위해 모두 같은 설정으로 새로 학습합니다. 이는 원래 실행에서 얻은 best.pt와 정확히 같은 파일을 재현한다는 의미는 아닙니다.*

In [ ]:
# 셀 1 — 패키지 설치, GPU 확인, Google Drive 연결
!pip -q install ultralytics pyyaml

from pathlib import Path
import zipfile, shutil, yaml, torch
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO
from google.colab import files, drive
from IPython.display import display

print('GPU 가능:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('런타임 → 런타임 유형 변경 → T4 GPU를 선택하세요.')
print('GPU:', torch.cuda.get_device_name(0))

drive.mount('/content/drive')
SAVE_DIR = Path('/content/drive/MyDrive/YOLO_3Models_Comparison')
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print('결과 저장 폴더:', SAVE_DIR)

In [ ]:
# 셀 2 — ZIP 3개를 한꺼번에 업로드
print('네 데이터셋, 상현 데이터셋, 지안 데이터셋 ZIP을 모두 선택하세요.')
uploaded = files.upload()
print('업로드:', list(uploaded.keys()))

def find_uploaded(prefix):
    matches = [name for name in uploaded if name.lower().startswith(prefix.lower())
               and name.lower().endswith('.zip')]
    if len(matches) != 1:
        raise FileNotFoundError(f'{prefix} ZIP 파일을 정확히 1개 업로드해야 합니다. 발견: {matches}')
    return Path('/content') / matches[0]

zip_files = {
    'mine': find_uploaded('my-bottle-detection'),
    'sanghyeon': find_uploaded('Yolo.v1i'),
    'jian': find_uploaded('Bottle Detection'),
}
print(zip_files)

In [ ]:
# 셀 3 — 압축 해제, data.yaml 절대경로 수정, 이미지 개수 확인
DATA_ROOT = Path('/content/yolo_3_datasets')
ORDER = ['mine', 'sanghyeon', 'jian']
DISPLAY_NAMES = {'mine':'Mine model', 'sanghyeon':'Sanghyeon model', 'jian':'Jian model'}
DATA_NAMES = {'mine':'Mine Test', 'sanghyeon':'Sanghyeon Test', 'jian':'Jian Test'}
data_yamls = {}

for person in ORDER:
    dest = DATA_ROOT / person
    if dest.exists():
        shutil.rmtree(dest)
    dest.mkdir(parents=True)
    with zipfile.ZipFile(zip_files[person]) as zf:
        zf.extractall(dest)

    yaml_path = dest / 'data.yaml'
    if not yaml_path.exists():
        raise FileNotFoundError(f'{person}: data.yaml이 없습니다: {dest}')
    with open(yaml_path, encoding='utf-8') as f:
        config = yaml.safe_load(f)

    config.update({
        'path': str(dest),
        'train': 'train/images',
        'val': 'valid/images',
        'test': 'test/images',
    })
    names = config.get('names')
    if names != ['my_bottle']:
        raise ValueError(f'{person}: 클래스 설정이 예상과 다릅니다. {names}')

    for split in ['train', 'val', 'test']:
        if not (dest / config[split]).is_dir():
            raise FileNotFoundError(f'{person}: {split}/images가 없습니다.')
    with open(yaml_path, 'w', encoding='utf-8') as f:
        yaml.safe_dump(config, f, allow_unicode=True, sort_keys=False)
    data_yamls[person] = str(yaml_path)
    counts = {s:len(list((dest/s/'images').glob('*.jpg'))) for s in ['train','valid','test']}
    print(f'{person}: {counts}, 클래스: {names}')

print('데이터셋 3개 준비 완료')

In [ ]:
# 셀 4 — 같은 사전학습 가중치에서 독립적으로 3개 모델 학습
# 저장된 best.pt를 매번 Drive에 복사하므로 Colab 연결이 끊겨도 보관할 수 있습니다.
RUN_ROOT = Path('/content/yolo_train_runs')
weights = {}

for person in ORDER:
    print(f'\n========== {person} 학습 ==========' )
    model = YOLO('yolo11n.pt')  # 각 모델마다 새 사전학습 모델에서 시작
    model.train(
        data=data_yamls[person],
        epochs=50, imgsz=640, batch=16, device=0,
        seed=42, deterministic=True, workers=2,
        project=str(RUN_ROOT), name=person, exist_ok=True, plots=True
    )
    best_file = RUN_ROOT / person / 'weights' / 'best.pt'
    if not best_file.exists():
        raise FileNotFoundError(f'학습 결과 best.pt 없음: {best_file}')

    saved_weight = SAVE_DIR / f'best_{person}.pt'
    shutil.copy2(best_file, saved_weight)
    weights[person] = str(saved_weight)
    print('모델 저장:', saved_weight)

print('3개 모델 준비 완료:', weights)

In [ ]:
# 셀 5 — 3개 모델 × 3개 Test 데이터 = 총 9번 교차 평가
# 주의: Test 분할에 대해 재학습하지 않습니다.
# Colab을 다시 켰더라도 셀 1~3을 실행했다면 Drive의 모델 파일을 읽을 수 있습니다.
weights = {person:str(SAVE_DIR / f'best_{person}.pt') for person in ORDER}
for path in weights.values():
    if not Path(path).exists():
        raise FileNotFoundError(f'모델 없음: {path} (셀 4부터 실행하세요)')

records = []
for model_name in ORDER:
    model = YOLO(weights[model_name])
    for test_name in ORDER:
        print(f'\n===== {model_name} 모델 → {test_name} Test =====')
        metrics = model.val(
            data=data_yamls[test_name], split='test', imgsz=640,
            batch=1, device=0, workers=2, plots=False, verbose=False,
            project='/content/yolo_cross_eval',
            name=f'{model_name}_on_{test_name}', exist_ok=True
        )
        records.append({
            'model':model_name, 'test_dataset':test_name,
            'Precision':float(metrics.box.mp),
            'Recall':float(metrics.box.mr),
            'mAP50':float(metrics.box.map50),
            'mAP50-95':float(metrics.box.map),
            'Inference_ms':float(metrics.speed['inference']),
        })
        pd.DataFrame(records).to_csv(SAVE_DIR / 'cross_dataset_results_partial.csv', index=False)

results_df = pd.DataFrame(records)
results_df.to_csv(SAVE_DIR / 'cross_dataset_results.csv', index=False)
print('총 평가 횟수:', len(results_df))
display(results_df.round(4))

In [ ]:
# 셀 6 — Self-Test Performance 그래프
# 각 모델이 자기 Test에 적용된 행만 추출
metric_cols = ['Precision', 'Recall', 'mAP50', 'mAP50-95']
self_df = results_df[results_df['model'] == results_df['test_dataset']].copy()
self_df = self_df.set_index('model').loc[ORDER, metric_cols] * 100
self_df.index = [DISPLAY_NAMES[x] for x in self_df.index]

ax = self_df.plot(kind='bar', figsize=(11,6), rot=0, width=0.78)
ax.set_title('Self-Test Performance (each model on its own Test set)')
ax.set_xlabel('Trained Model')
ax.set_ylabel('Score (%)')
ax.set_ylim(0, 108)
ax.grid(axis='y', alpha=0.25)
ax.legend(loc='lower right')
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f', padding=2, fontsize=8)
plt.tight_layout()
plt.savefig(SAVE_DIR / 'self_test_performance.png', dpi=220, bbox_inches='tight')
plt.show()

In [ ]:
# 셀 7 — Cross-Dataset mAP50-95 Heatmap
mat = results_df.pivot(index='model', columns='test_dataset', values='mAP50-95')
mat = mat.reindex(index=ORDER, columns=ORDER) * 100
fig, ax = plt.subplots(figsize=(8,6))
im = ax.imshow(mat.values, cmap='Blues', vmin=0, vmax=100)
ax.set_xticks(range(3), [DATA_NAMES[x] for x in ORDER])
ax.set_yticks(range(3), [DISPLAY_NAMES[x] for x in ORDER])
ax.set_xlabel('Test Dataset')
ax.set_ylabel('Trained Model')
ax.set_title('Cross-Dataset Evaluation (mAP50-95, %)')

for row in range(3):
    for col in range(3):
        value = mat.iat[row, col]
        ax.text(col, row, f'{value:.1f}', ha='center', va='center',
                color='white' if value >= 55 else 'black', fontsize=12, fontweight='bold')
fig.colorbar(im, ax=ax, label='mAP50-95 (%)')
plt.tight_layout()
plt.savefig(SAVE_DIR / 'cross_dataset_heatmap.png', dpi=220, bbox_inches='tight')
plt.show()

In [ ]:
# 셀 8 — 평가 표 / 그래프 / 3개 모델을 ZIP으로 다운로드
package = Path('/content/YOLO_3Models_Comparison_Results.zip')
file_names = [
    'cross_dataset_results.csv',
    'self_test_performance.png',
    'cross_dataset_heatmap.png',
    'best_mine.pt', 'best_sanghyeon.pt', 'best_jian.pt',
]
with zipfile.ZipFile(package, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for filename in file_names:
        path = SAVE_DIR / filename
        if not path.exists():
            raise FileNotFoundError(f'다운로드 파일이 없습니다: {path}')
        zf.write(path, arcname=filename)
print('Google Drive 저장 폴더:', SAVE_DIR)
print('다운로드 파일:', package)
files.download(str(package))